loading dataset from kaggle itself(after adding it to input )

In [2]:
import os

# Sare mounted datasets dekho
print(os.listdir("/kaggle/input"))

['datasets']


In [5]:
import os

print(os.listdir("/kaggle/input/datasets/dorianlazar/medium-articles-dataset"))

['medium_data.csv', 'images']


In [7]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/dorianlazar/medium-articles-dataset/medium_data.csv")

**IMPORTING LIBRARIES**

In [9]:
!pip install nltk

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [8]:
df.head()

,id,url,title,subtitle,image,claps,responses,reading_time,publication,date
0,1,https://towardsdatascience.com/a-beginners-gui...,A Beginner’s Guide to Word Embedding with Gens...,NaN,1.png,850,8,8,Towards Data Science,2019-05-30
1,2,https://towardsdatascience.com/hands-on-graph-...,Hands-on Graph Neural Networks with PyTorch & ...,NaN,2.png,1100,11,9,Towards Data Science,2019-05-30
2,3,https://towardsdatascience.com/how-to-use-ggpl...,How to Use ggplot2 in Python,A Grammar of Graphics for Python,3.png,767,1,5,Towards Data Science,2019-05-30
3,4,https://towardsdatascience.com/databricks-how-...,Databricks: How to Save Files in CSV on Your L...,When I work on Python projects dealing…,4.jpeg,354,0,4,Towards Data Science,2019-05-30
4,5,https://towardsdatascience.com/a-step-by-step-...,A Step-by-Step Implementation of Gradient Desc...,One example of building neural…,5.jpeg,211,3,4,Towards Data Science,2019-05-30


preparing the training data 

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6508 entries, 0 to 6507
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            6508 non-null   int64 
 1   url           6508 non-null   object
 2   title         6508 non-null   object
 3   subtitle      3479 non-null   object
 4   image         6361 non-null   object
 5   claps         6508 non-null   int64 
 6   responses     6508 non-null   object
 7   reading_time  6508 non-null   int64 
 8   publication   6508 non-null   object
 9   date          6508 non-null   object
dtypes: int64(3), object(7)
memory usage: 508.6+ KB


In [12]:
# Convert df['title'] into a single string
document = '\n'.join(df['title'].dropna().astype(str))

In [13]:
document

'A Beginner’s Guide to Word Embedding with Gensim Word2Vec\xa0Model\nHands-on Graph Neural Networks with PyTorch & PyTorch Geometric\nHow to Use ggplot2 in\xa0Python\nDatabricks: How to Save Files in CSV on Your Local\xa0Computer\nA Step-by-Step Implementation of Gradient Descent and Backpropagation\nAn Easy Introduction to SQL for Data Scientists\nHypothesis testing visualized\nIntroduction to Latent Matrix Factorization Recommender Systems\nWhich 2020 Candidate is the Best at\xa0Twitter?\nWhat if AI model understanding were\xa0easy?\n<em class="markup--em markup--h3-em">What I Learned from (Two-time) Kaggle Grandmaster Abhishek\xa0Thakur</em>\nMaking a DotA2 Bot Using\xa0ML\nBuilding A ‘Serverless’ Chrome Extension\nHow to Teach\xa0Code\nReinventing Personalization For Customer Experience\nHow to Automate Hyperparameter Optimization\nIdeas: Design Methodologies for Data\xa0Sprints\nRoboSomm Chapter 3: Wine Embeddings and a Wine Recommender\nData Science Interview Questions\nFaster Tr

In [14]:
# Tokenization
nltk.download('punkt')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [15]:
tokens = word_tokenize(document.lower())

In [16]:
tokens

['a',
 'beginner',
 '’',
 's',
 'guide',
 'to',
 'word',
 'embedding',
 'with',
 'gensim',
 'word2vec',
 'model',
 'hands-on',
 'graph',
 'neural',
 'networks',
 'with',
 'pytorch',
 '&',
 'pytorch',
 'geometric',
 'how',
 'to',
 'use',
 'ggplot2',
 'in',
 'python',
 'databricks',
 ':',
 'how',
 'to',
 'save',
 'files',
 'in',
 'csv',
 'on',
 'your',
 'local',
 'computer',
 'a',
 'step-by-step',
 'implementation',
 'of',
 'gradient',
 'descent',
 'and',
 'backpropagation',
 'an',
 'easy',
 'introduction',
 'to',
 'sql',
 'for',
 'data',
 'scientists',
 'hypothesis',
 'testing',
 'visualized',
 'introduction',
 'to',
 'latent',
 'matrix',
 'factorization',
 'recommender',
 'systems',
 'which',
 '2020',
 'candidate',
 'is',
 'the',
 'best',
 'at',
 'twitter',
 '?',
 'what',
 'if',
 'ai',
 'model',
 'understanding',
 'were',
 'easy',
 '?',
 '<',
 'em',
 'class=',
 "''",
 'markup',
 '--',
 'em',
 'markup',
 '--',
 'h3-em',
 "''",
 '>',
 'what',
 'i',
 'learned',
 'from',
 '(',
 'two-time',

In [17]:
# build vocab
vocab = {'<unk>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

vocab

{'<unk>': 0,
 'a': 1,
 'beginner': 2,
 '’': 3,
 's': 4,
 'guide': 5,
 'to': 6,
 'word': 7,
 'embedding': 8,
 'with': 9,
 'gensim': 10,
 'word2vec': 11,
 'model': 12,
 'hands-on': 13,
 'graph': 14,
 'neural': 15,
 'networks': 16,
 'pytorch': 17,
 '&': 18,
 'geometric': 19,
 'how': 20,
 'use': 21,
 'ggplot2': 22,
 'in': 23,
 'python': 24,
 'databricks': 25,
 ':': 26,
 'save': 27,
 'files': 28,
 'csv': 29,
 'on': 30,
 'your': 31,
 'local': 32,
 'computer': 33,
 'step-by-step': 34,
 'implementation': 35,
 'of': 36,
 'gradient': 37,
 'descent': 38,
 'and': 39,
 'backpropagation': 40,
 'an': 41,
 'easy': 42,
 'introduction': 43,
 'sql': 44,
 'for': 45,
 'data': 46,
 'scientists': 47,
 'hypothesis': 48,
 'testing': 49,
 'visualized': 50,
 'latent': 51,
 'matrix': 52,
 'factorization': 53,
 'recommender': 54,
 'systems': 55,
 'which': 56,
 '2020': 57,
 'candidate': 58,
 'is': 59,
 'the': 60,
 'best': 61,
 'at': 62,
 'twitter': 63,
 '?': 64,
 'what': 65,
 'if': 66,
 'ai': 67,
 'understanding': 

In [18]:
len(vocab)

8347

In [19]:
# Break your document in sentences
input_sentences = document.split('\n')

In [20]:
input_sentences

['A Beginner’s Guide to Word Embedding with Gensim Word2Vec\xa0Model',
 'Hands-on Graph Neural Networks with PyTorch & PyTorch Geometric',
 'How to Use ggplot2 in\xa0Python',
 'Databricks: How to Save Files in CSV on Your Local\xa0Computer',
 'A Step-by-Step Implementation of Gradient Descent and Backpropagation',
 'An Easy Introduction to SQL for Data Scientists',
 'Hypothesis testing visualized',
 'Introduction to Latent Matrix Factorization Recommender Systems',
 'Which 2020 Candidate is the Best at\xa0Twitter?',
 'What if AI model understanding were\xa0easy?',
 '<em class="markup--em markup--h3-em">What I Learned from (Two-time) Kaggle Grandmaster Abhishek\xa0Thakur</em>',
 'Making a DotA2 Bot Using\xa0ML',
 'Building A ‘Serverless’ Chrome Extension',
 'How to Teach\xa0Code',
 'Reinventing Personalization For Customer Experience',
 'How to Automate Hyperparameter Optimization',
 'Ideas: Design Methodologies for Data\xa0Sprints',
 'RoboSomm Chapter 3: Wine Embeddings and a Wine Reco

In [21]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence

In [22]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))


In [23]:
len(input_numerical_sentences)

6508

In [25]:
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [26]:
training_sequence

[[1, 2],
 [1, 2, 3],
 [1, 2, 3, 4],
 [1, 2, 3, 4, 5],
 [1, 2, 3, 4, 5, 6],
 [1, 2, 3, 4, 5, 6, 7],
 [1, 2, 3, 4, 5, 6, 7, 8],
 [1, 2, 3, 4, 5, 6, 7, 8, 9],
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
 [13, 14],
 [13, 14, 15],
 [13, 14, 15, 16],
 [13, 14, 15, 16, 9],
 [13, 14, 15, 16, 9, 17],
 [13, 14, 15, 16, 9, 17, 18],
 [13, 14, 15, 16, 9, 17, 18, 17],
 [13, 14, 15, 16, 9, 17, 18, 17, 19],
 [20, 6],
 [20, 6, 21],
 [20, 6, 21, 22],
 [20, 6, 21, 22, 23],
 [20, 6, 21, 22, 23, 24],
 [25, 26],
 [25, 26, 20],
 [25, 26, 20, 6],
 [25, 26, 20, 6, 27],
 [25, 26, 20, 6, 27, 28],
 [25, 26, 20, 6, 27, 28, 23],
 [25, 26, 20, 6, 27, 28, 23, 29],
 [25, 26, 20, 6, 27, 28, 23, 29, 30],
 [25, 26, 20, 6, 27, 28, 23, 29, 30, 31],
 [25, 26, 20, 6, 27, 28, 23, 29, 30, 31, 32],
 [25, 26, 20, 6, 27, 28, 23, 29, 30, 31, 32, 33],
 [1, 34],
 [1, 34, 35],
 [1, 34, 35, 36],
 [1, 34, 35, 36, 37],
 [1, 34, 35, 36, 37, 38],
 [1, 34, 35, 36, 37, 38

In [27]:
len(training_sequence)

55467

In [28]:
training_sequence[:5]

[[1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]]

In [29]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

51

In [30]:
training_sequence[0]

[1, 2]

In [31]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [32]:
len(padded_training_sequence[0])

51

In [33]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

In [34]:
padded_training_sequence[:3]

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 1, 2],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         1, 2, 3],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
         2, 3, 4]])

In [38]:
# Divide the data in X and Y
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]

In [39]:
X

tensor([[   0,    0,    0,  ...,    0,    0,    1],
        [   0,    0,    0,  ...,    0,    1,    2],
        [   0,    0,    0,  ...,    1,    2,    3],
        ...,
        [   0,    0,    0,  ...,  677,    1,  551],
        [   0,    0,    0,  ...,    1,  551,  303],
        [   0,    0,    0,  ...,  551,  303, 2870]])

In [40]:
y

tensor([   2,    3,    4,  ...,  303, 2870, 2403])

**DATASET AND DATALOADER**

In [41]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [42]:
dataset = CustomDataset(X,y)

In [43]:
len(dataset)

55467

In [45]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

**LSTM model**** **

In [47]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    # You want to convert that 150-dimensional hidden state into a probability distribution over the entire vocabulary.
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    # why you squeeze ??
    # Since your LSTM has:
    # 1 layer, not bidirectional,
    # You get: final_hidden_state.shape = (1, batch_size, 150)
    return output

# intermediate_hidden_states: output at each time step → shape: (batch_size, seq_len, hidden_size)


In [49]:
model = LSTMModel(len(vocab))

In [53]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [54]:
model.to(device)

LSTMModel(
  (embedding): Embedding(8347, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=8347, bias=True)
)

In [55]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

**TRAINING**

In [56]:
# Training loop for multiple epochs
for epoch in range(epochs):
  total_loss = 0  # Initialize total loss for this epoch

  # Iterate over batches from the dataloader
  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)  # Move data to GPU or CPU

    optimizer.zero_grad()  # Clear previous gradients

    output = model(batch_x)  # Forward pass through the model

    loss = criterion(output, batch_y)  # Compute loss between predicted and target values

    loss.backward()  # Backward pass to compute gradients

    optimizer.step()  # Update model parameters using gradients

    total_loss = total_loss + loss.item()  # Accumulate loss for monitoring

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")  # Print loss after each epoch


Epoch: 1, Loss: 11011.5016
Epoch: 2, Loss: 9200.3087
Epoch: 3, Loss: 8056.6178
Epoch: 4, Loss: 7014.4244
Epoch: 5, Loss: 6075.9115
Epoch: 6, Loss: 5246.8971
Epoch: 7, Loss: 4537.8826
Epoch: 8, Loss: 3944.8336
Epoch: 9, Loss: 3444.4518
Epoch: 10, Loss: 3018.9591
Epoch: 11, Loss: 2665.8458
Epoch: 12, Loss: 2358.6066
Epoch: 13, Loss: 2102.2888
Epoch: 14, Loss: 1886.5927
Epoch: 15, Loss: 1707.0559
Epoch: 16, Loss: 1557.5175
Epoch: 17, Loss: 1435.7758
Epoch: 18, Loss: 1335.1847
Epoch: 19, Loss: 1258.2963
Epoch: 20, Loss: 1191.9934
Epoch: 21, Loss: 1138.8726
Epoch: 22, Loss: 1099.0421
Epoch: 23, Loss: 1065.6250
Epoch: 24, Loss: 1037.2260
Epoch: 25, Loss: 1014.5304
Epoch: 26, Loss: 998.9749
Epoch: 27, Loss: 986.2925
Epoch: 28, Loss: 971.2915
Epoch: 29, Loss: 961.4259
Epoch: 30, Loss: 952.5123
Epoch: 31, Loss: 945.0430
Epoch: 32, Loss: 934.7130
Epoch: 33, Loss: 932.1980
Epoch: 34, Loss: 923.4205
Epoch: 35, Loss: 920.8435
Epoch: 36, Loss: 917.0786
Epoch: 37, Loss: 913.2270
Epoch: 38, Loss: 905.

**MODEL TESTING**

In [57]:
import torch
from nltk.tokenize import word_tokenize

def prediction(model, vocab, text):
    # Choose the device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Move model to the device
    model = model.to(device)

    # Tokenize
    tokenized_text = word_tokenize(text.lower())

    # Text -> indices
    numerical_text = text_to_indices(tokenized_text, vocab)

    # Padding (assuming 51 is the fixed input length)
    padded_text = torch.tensor([0] * (51 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

    # Move input to device
    padded_text = padded_text.to(device)

    # Model prediction
    output = model(padded_text)

    # Get predicted index
    value , index = torch.max(output, dim=1)

    # Get predicted token (label, etc.)
    predicted_token = list(vocab.keys())[index]

    return text + " " + predicted_token


In [58]:
prediction(model, vocab, "Databricks: How to Save Files in")

'Databricks: How to Save Files in csv'

In [59]:
prediction(model, vocab, "A Step-by-Step Implementation of")

'A Step-by-Step Implementation of gradient'

In [60]:
import time

num_tokens = 20
input_text = "A Step-by-Step Implementation of"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)


A Step-by-Step Implementation of gradient
A Step-by-Step Implementation of gradient descent
A Step-by-Step Implementation of gradient descent and
A Step-by-Step Implementation of gradient descent and backpropagation
A Step-by-Step Implementation of gradient descent and backpropagation ,
A Step-by-Step Implementation of gradient descent and backpropagation , accessibility
A Step-by-Step Implementation of gradient descent and backpropagation , accessibility ,
A Step-by-Step Implementation of gradient descent and backpropagation , accessibility , and
A Step-by-Step Implementation of gradient descent and backpropagation , accessibility , and intuition
A Step-by-Step Implementation of gradient descent and backpropagation , accessibility , and intuition (
A Step-by-Step Implementation of gradient descent and backpropagation , accessibility , and intuition ( guide
A Step-by-Step Implementation of gradient descent and backpropagation , accessibility , and intuition ( guide to
A Step-by-Step Im